In [ ]:
# @title 1. Install, imports, Drive mount, device check

%pip install -q -U flax optax tensorflow tensorflow-datasets pandas

import os
import gc
import re
import json
import time
import random as py_random
from pathlib import Path
from typing import Any, Dict
from functools import partial

import numpy as np
import pandas as pd

import jax
import jax.numpy as jnp
from jax import random, lax

import flax
import flax.linen as nn
from flax import serialization, jax_utils, traverse_util
from flax.training import train_state

import optax

import tensorflow as tf
import tensorflow_datasets as tfds

from google.colab import drive
from tqdm.auto import tqdm

tf.get_logger().setLevel("ERROR")
drive.mount("/content/drive", force_remount=True)

NUM_DEVICES = jax.local_device_count()

print("JAX version:", jax.__version__)
print("Backend:", jax.default_backend())
print("Devices:", jax.devices())
print("NUM_DEVICES:", NUM_DEVICES)

if NUM_DEVICES < 1:
    raise RuntimeError("No JAX devices found.")

In [ ]:
# @title 2. Global config, save layout, robust-bank manifest

DRIVE_ROOT = Path("/content/drive/MyDrive")

# Same root address as your earlier bank
RUN_ROOT = DRIVE_ROOT / "representation_bank"
RUN_GROUP = "adv_resnet18_cifar_v1"

TFDS_DATA_DIR = DRIVE_ROOT / "tfds_data"

RUN_ROOT.mkdir(parents=True, exist_ok=True)
TFDS_DATA_DIR.mkdir(parents=True, exist_ok=True)
# The submitted paper reports CIFAR-10 robust-training experiments.
DATASET_NAMES = ["cifar10", "cifar100"]
MODEL_NAMES = ["resnet18_pgd", "resnet18_trades", "resnet18_mart"]
SEEDS = list(range(5))

DATASET_CFG = {
    "cifar10": {
        "num_classes": 10,
        "image_size": 32,
        "train_split": "train",
        "eval_split": "test",
        "mean": (0.4914, 0.4822, 0.4465),
        "std":  (0.2470, 0.2435, 0.2616),
    },
    "cifar100": {
        "num_classes": 100,
        "image_size": 32,
        "train_split": "train",
        "eval_split": "test",
        "mean": (0.5071, 0.4867, 0.4408),
        "std":  (0.2675, 0.2565, 0.2761),
    },
}

# Robust CIFAR defaults
# Epsilon = 8/255, step = 2/255, 10-step train PGD, 20-step eval PGD
ATTACK_CFG = {
    "epsilon": 8.0 / 255.0,
    "alpha_train": 2.0 / 255.0,
    "alpha_eval": 2.0 / 255.0,
    "pgd_steps_train": 10,
    "pgd_steps_eval": 20,
}

# One robust bank architecture, three defenses
# model_name encodes the defense to preserve the same 6-arg loader API
TRAINING_CFG = {
    ("cifar10", "resnet18_pgd"): dict(
        epochs=110, batch_size=256, lr=0.1, weight_decay=5e-4, momentum=0.9,
        beta=None, choose_best_by="robust_acc"
    ),
    ("cifar100", "resnet18_pgd"): dict(
        epochs=110, batch_size=256, lr=0.1, weight_decay=5e-4, momentum=0.9,
        beta=None, choose_best_by="robust_acc"
    ),
    ("cifar10", "resnet18_trades"): dict(
        epochs=110, batch_size=256, lr=0.1, weight_decay=5e-4, momentum=0.9,
        beta=5.0, choose_best_by="robust_acc"
    ),
    ("cifar100", "resnet18_trades"): dict(
        epochs=110, batch_size=256, lr=0.1, weight_decay=5e-4, momentum=0.9,
        beta=5.0, choose_best_by="robust_acc"
    ),
    ("cifar10", "resnet18_mart"): dict(
        epochs=110, batch_size=256, lr=0.1, weight_decay=5e-4, momentum=0.9,
        beta=5.0, choose_best_by="robust_acc"
    ),
    ("cifar100", "resnet18_mart"): dict(
        epochs=110, batch_size=256, lr=0.1, weight_decay=5e-4, momentum=0.9,
        beta=5.0, choose_best_by="robust_acc"
    ),
}

def natural_key(s: str):
    return [int(x) if x.isdigit() else x for x in re.split(r"(\d+)", s)]

def parse_model_name(model_name: str):
    if not model_name.startswith("resnet18_"):
        raise ValueError(f"Unexpected model_name: {model_name}")
    arch = "resnet18"
    defense = model_name.split("_", 1)[1]
    return arch, defense

def set_all_seeds(seed: int):
    np.random.seed(seed)
    py_random.seed(seed)
    tf.random.set_seed(seed)

def run_dir(save_root, run_group, dataset_name, model_name, seed):
    return Path(save_root) / run_group / dataset_name / model_name / f"seed_{seed:02d}"

def save_json(path: Path, obj: Dict[str, Any]):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

def load_json(path: Path):
    with open(path, "r") as f:
        return json.load(f)

def shard_array(x: np.ndarray):
    n = x.shape[0]
    if n % NUM_DEVICES != 0:
        raise ValueError(f"Batch size {n} is not divisible by NUM_DEVICES={NUM_DEVICES}")
    return x.reshape((NUM_DEVICES, n // NUM_DEVICES) + x.shape[1:])

def make_batch_dict(batch):
    images, labels = batch
    return {
        "images": shard_array(images),
        "labels": shard_array(labels),
    }

def numpy_prefetch_iter(tf_dataset, prefetch_size=2):
    generator = (make_batch_dict(batch) for batch in tfds.as_numpy(tf_dataset))
    return jax_utils.prefetch_to_device(generator, prefetch_size)

print("RUN_ROOT:", RUN_ROOT)
print("RUN_GROUP:", RUN_GROUP)
print("TFDS_DATA_DIR:", TFDS_DATA_DIR)
print("DATASET_NAMES:", DATASET_NAMES)
print("MODEL_NAMES:", MODEL_NAMES)

In [ ]:
# @title 3. CIFAR data pipeline

AUTOTUNE = tf.data.AUTOTUNE

def normalize_image(image, mean, std):
    image = tf.cast(image, tf.float32) / 255.0
    mean = tf.constant(mean, dtype=tf.float32)[None, None, :]
    std = tf.constant(std, dtype=tf.float32)[None, None, :]
    return (image - mean) / std

def preprocess_cifar_train(idx, image, label, dataset_name, seed):
    cfg = DATASET_CFG[dataset_name]
    image = tf.image.resize_with_crop_or_pad(image, 40, 40)
    seed_pair = tf.stack([
        tf.cast(seed, tf.int32),
        tf.cast(idx % 2_000_000_000, tf.int32),
    ])
    image = tf.image.stateless_random_crop(image, size=(32, 32, 3), seed=seed_pair)
    image = tf.image.stateless_random_flip_left_right(
        image, seed=seed_pair + tf.constant([0, 1], dtype=tf.int32)
    )
    image = normalize_image(image, cfg["mean"], cfg["std"])
    return image, tf.cast(label, tf.int32)

def preprocess_cifar_eval(image, label, dataset_name):
    cfg = DATASET_CFG[dataset_name]
    image = normalize_image(image, cfg["mean"], cfg["std"])
    return image, tf.cast(label, tf.int32)

def build_cifar_dataset(dataset_name, split, batch_size, training, seed):
    builder = tfds.builder(dataset_name, data_dir=str(TFDS_DATA_DIR))
    builder.download_and_prepare()

    ds = builder.as_dataset(
        split=split,
        as_supervised=True,
        shuffle_files=training,
        read_config=tfds.ReadConfig(shuffle_seed=seed),
    )

    if training:
        ds = ds.cache()
        ds = ds.shuffle(50_000, seed=seed, reshuffle_each_iteration=True)
        ds = ds.repeat()
        ds = ds.enumerate()
        ds = ds.map(
            lambda i, xy: preprocess_cifar_train(i, xy[0], xy[1], dataset_name, seed),
            num_parallel_calls=AUTOTUNE,
        )
    else:
        ds = ds.map(
            lambda x, y: preprocess_cifar_eval(x, y, dataset_name),
            num_parallel_calls=AUTOTUNE,
        )

    ds = ds.batch(batch_size, drop_remainder=True)
    ds = ds.prefetch(AUTOTUNE)

    num_examples = builder.info.splits[split].num_examples
    class_names = builder.info.features["label"].names
    return ds, num_examples, class_names

def make_dataloaders(dataset_name, batch_size, seed):
    if batch_size % NUM_DEVICES != 0:
        raise ValueError(f"batch_size={batch_size} must be divisible by NUM_DEVICES={NUM_DEVICES}")

    train_ds, train_size, class_names = build_cifar_dataset(
        dataset_name=dataset_name,
        split=DATASET_CFG[dataset_name]["train_split"],
        batch_size=batch_size,
        training=True,
        seed=seed,
    )
    eval_ds, eval_size, _ = build_cifar_dataset(
        dataset_name=dataset_name,
        split=DATASET_CFG[dataset_name]["eval_split"],
        batch_size=batch_size,
        training=False,
        seed=seed,
    )

    steps_per_epoch = train_size // batch_size
    eval_steps = eval_size // batch_size
    train_iter = iter(numpy_prefetch_iter(train_ds, prefetch_size=2))
    return train_iter, eval_ds, steps_per_epoch, eval_steps, class_names

print("Data pipeline ready.")

In [ ]:
# @title 4. ResNet-18 definition with saved intermediate activations

class TrainState(train_state.TrainState):
    batch_stats: Any = flax.struct.field(pytree_node=True, default_factory=dict)

class ResidualBlock(nn.Module):
    features: int
    stride: int = 1

    @nn.compact
    def __call__(self, x, train: bool):
        residual = x

        y = nn.Conv(
            self.features, (3, 3),
            strides=(self.stride, self.stride),
            padding="SAME",
            use_bias=False,
        )(x)
        y = nn.BatchNorm(
            use_running_average=not train,
            momentum=0.9,
            epsilon=1e-5,
        )(y)
        y = nn.relu(y)

        y = nn.Conv(
            self.features, (3, 3),
            strides=(1, 1),
            padding="SAME",
            use_bias=False,
        )(y)
        y = nn.BatchNorm(
            use_running_average=not train,
            momentum=0.9,
            epsilon=1e-5,
            scale_init=nn.initializers.zeros,
        )(y)

        if residual.shape != y.shape:
            residual = nn.Conv(
                self.features, (1, 1),
                strides=(self.stride, self.stride),
                use_bias=False,
            )(residual)
            residual = nn.BatchNorm(
                use_running_average=not train,
                momentum=0.9,
                epsilon=1e-5,
            )(residual)

        return nn.relu(residual + y)

class ResNet18(nn.Module):
    num_classes: int
    image_size: int = 32

    @nn.compact
    def __call__(self, x, train: bool):
        x = nn.Conv(
            64, (3, 3),
            strides=(1, 1),
            padding="SAME",
            use_bias=False,
            name="stem_conv",
        )(x)
        x = nn.BatchNorm(
            use_running_average=not train,
            momentum=0.9,
            epsilon=1e-5,
            name="stem_bn",
        )(x)
        x = nn.relu(x)
        self.sow("intermediates", "act_stem", x)

        block_specs = [
            (64, 1), (64, 1),
            (128, 2), (128, 1),
            (256, 2), (256, 1),
            (512, 2), (512, 1),
        ]

        for i, (features, stride) in enumerate(block_specs, start=1):
            x = ResidualBlock(features=features, stride=stride, name=f"block{i}")(x, train=train)
            self.sow("intermediates", f"act_block{i}", x)

        x = jnp.mean(x, axis=(1, 2))
        self.sow("intermediates", "act_pre_logits", x)
        logits = nn.Dense(self.num_classes, name="head")(x)
        return logits

def build_model(model_name: str, dataset_name: str):
    arch, defense = parse_model_name(model_name)
    if arch != "resnet18":
        raise ValueError(model_name)

    num_classes = DATASET_CFG[dataset_name]["num_classes"]
    model = ResNet18(num_classes=num_classes, image_size=32)
    model_cfg = {
        "model_name": model_name,
        "arch": arch,
        "defense_method": defense,
        "num_classes": num_classes,
        "image_size": 32,
    }
    return model, model_cfg

def get_layer_names(model, dataset_name):
    image_size = DATASET_CFG[dataset_name]["image_size"]
    dummy = jnp.zeros((1, image_size, image_size, 3), dtype=jnp.float32)
    key = random.PRNGKey(0)
    vars0 = model.init({"params": key}, dummy, train=False)
    _, mut = model.apply(vars0, dummy, train=False, mutable=["intermediates"])
    layer_names = sorted(mut["intermediates"].keys(), key=natural_key)
    return list(layer_names)

print("Model definition ready.")

In [ ]:
# @title 5. Robust-training losses, attacks, optimizer, eval, checkpoint helpers

def dataset_attack_tensors(dataset_name):
    cfg = DATASET_CFG[dataset_name]
    mean = jnp.array(cfg["mean"], dtype=jnp.float32).reshape(1, 1, 1, 3)
    std = jnp.array(cfg["std"], dtype=jnp.float32).reshape(1, 1, 1, 3)

    lower = (0.0 - mean) / std
    upper = (1.0 - mean) / std

    eps = (ATTACK_CFG["epsilon"] / std)
    alpha_train = (ATTACK_CFG["alpha_train"] / std)
    alpha_eval = (ATTACK_CFG["alpha_eval"] / std)

    return lower, upper, eps, alpha_train, alpha_eval

def clip_project(x_adv, x0, lower, upper, eps):
    x_adv = jnp.minimum(jnp.maximum(x_adv, x0 - eps), x0 + eps)
    x_adv = jnp.clip(x_adv, lower, upper)
    return x_adv

def kl_divergence_probs_logits(target_probs, logits):
    log_target = jnp.log(jnp.clip(target_probs, 1e-12, 1.0))
    log_pred = jax.nn.log_softmax(logits, axis=-1)
    return jnp.sum(target_probs * (log_target - log_pred), axis=-1)

def mart_loss(logits_clean, logits_adv, labels, beta):
    adv_probs = jax.nn.softmax(logits_adv, axis=-1)

    top2 = jnp.argsort(adv_probs, axis=1)[:, -2:]
    top1 = top2[:, 1]
    second = top2[:, 0]
    new_y = jnp.where(top1 == labels, second, top1)

    ce_adv = optax.softmax_cross_entropy_with_integer_labels(logits_adv, labels)
    log_1m_adv = jnp.log(jnp.clip(1.0001 - adv_probs, 1e-12, 1.0))
    nll_margin = -jnp.take_along_axis(log_1m_adv, new_y[:, None], axis=1).squeeze(1)
    loss_adv = ce_adv + nll_margin

    nat_probs = jax.nn.softmax(logits_clean, axis=-1)
    true_probs = jnp.take_along_axis(nat_probs, labels[:, None], axis=1).squeeze(1)
    kl_term = kl_divergence_probs_logits(nat_probs, logits_adv)
    loss_rob = kl_term * (1.0 - true_probs)

    return jnp.mean(loss_adv + beta * loss_rob)

def decay_mask_from_params(params):
    flat = traverse_util.flatten_dict(params)
    mask = {}
    for key_tuple, value in flat.items():
        key = "/".join(key_tuple)
        use_decay = getattr(value, "ndim", 0) > 1
        if any(tok in key for tok in ["bias", "scale", "BatchNorm", "batchnorm"]):
            use_decay = False
        mask[key_tuple] = use_decay
    return traverse_util.unflatten_dict(mask)

def make_lr_schedule(base_lr, steps_per_epoch, epochs):
    # 110-epoch standard-ish multistep schedule: 55 / 75 / 90
    boundaries_and_scales = {
        55 * steps_per_epoch: 0.1,
        75 * steps_per_epoch: 0.1,
        90 * steps_per_epoch: 0.1,
    }
    return optax.piecewise_constant_schedule(
        init_value=base_lr,
        boundaries_and_scales=boundaries_and_scales,
    )

def init_state_for_run(model, dataset_name, model_name, seed, steps_per_epoch):
    sched = TRAINING_CFG[(dataset_name, model_name)]
    image_size = DATASET_CFG[dataset_name]["image_size"]
    lr_fn = make_lr_schedule(sched["lr"], steps_per_epoch, sched["epochs"])

    rng = random.PRNGKey(seed)
    dummy = jnp.zeros((1, image_size, image_size, 3), dtype=jnp.float32)
    variables = model.init({"params": rng}, dummy, train=True)

    params = variables["params"]
    batch_stats = variables.get("batch_stats", {})
    mask = decay_mask_from_params(params)

    tx = optax.chain(
        optax.add_decayed_weights(sched["weight_decay"], mask=mask),
        optax.sgd(
            learning_rate=lr_fn,
            momentum=sched["momentum"],
            nesterov=True,
        ),
    )

    state = TrainState.create(
        apply_fn=model.apply,
        params=params,
        tx=tx,
        batch_stats=batch_stats,
    )
    return state, lr_fn

def pgd_ce_attack(apply_fn, params, batch_stats, x, y, rng, dataset_name, steps, alpha):
    lower, upper, eps, alpha_train, alpha_eval = dataset_attack_tensors(dataset_name)
    step_size = alpha_train if alpha == "train" else alpha_eval

    noise = random.uniform(rng, shape=x.shape, minval=-1.0, maxval=1.0) * eps
    x_adv = jnp.clip(x + noise, lower, upper)

    variables = {"params": params, "batch_stats": batch_stats}

    def loss_on_x(x_in):
        logits = apply_fn(variables, x_in, train=False)
        return optax.softmax_cross_entropy_with_integer_labels(logits, y).mean()

    def body_fn(i, x_adv_now):
        grad = jax.grad(loss_on_x)(x_adv_now)
        x_next = x_adv_now + step_size * jnp.sign(grad)
        x_next = clip_project(x_next, x, lower, upper, eps)
        return x_next

    return lax.fori_loop(0, steps, body_fn, x_adv)

# @title PATCH 1: replace trades_attack with this version

def trades_attack(apply_fn, params, batch_stats, x, nat_probs, rng, dataset_name, steps):
    lower, upper, eps, alpha_train, _ = dataset_attack_tensors(dataset_name)

    # Official TRADES uses a tiny Gaussian init, not a full-epsilon random start.
    std = jnp.array(DATASET_CFG[dataset_name]["std"], dtype=jnp.float32).reshape(1, 1, 1, 3)
    noise_scale = 0.001 / std
    noise = random.normal(rng, shape=x.shape) * noise_scale
    x_adv = jnp.clip(x + noise, lower, upper)

    variables = {"params": params, "batch_stats": batch_stats}
    nat_probs = lax.stop_gradient(nat_probs)

    def loss_on_x(x_in):
        logits_adv = apply_fn(variables, x_in, train=False)
        return kl_divergence_probs_logits(nat_probs, logits_adv).mean()

    def body_fn(i, x_adv_now):
        grad = jax.grad(loss_on_x)(x_adv_now)
        x_next = x_adv_now + alpha_train * jnp.sign(grad)
        x_next = clip_project(x_next, x, lower, upper, eps)
        return x_next

    x_adv = lax.fori_loop(0, steps, body_fn, x_adv)
    return x_adv


# @title PATCH 2: replace make_train_step with this version

def make_train_step(dataset_name, model_name):
    _, defense = parse_model_name(model_name)
    beta = TRAINING_CFG[(dataset_name, model_name)]["beta"]

    @partial(jax.pmap, axis_name="batch", donate_argnums=(0,))
    def train_step(state, batch, rng):
        x = batch["images"]
        y = batch["labels"]

        attack_rng, _ = random.split(rng)

        frozen_vars = {"params": state.params, "batch_stats": state.batch_stats}

        if defense == "pgd":
            x_adv = pgd_ce_attack(
                apply_fn=state.apply_fn,
                params=state.params,
                batch_stats=state.batch_stats,
                x=x,
                y=y,
                rng=attack_rng,
                dataset_name=dataset_name,
                steps=ATTACK_CFG["pgd_steps_train"],
                alpha="train",
            )

            def loss_fn(params):
                variables = {"params": params, "batch_stats": state.batch_stats}
                logits_adv, mut = state.apply_fn(
                    variables, x_adv, train=True, mutable=["batch_stats"]
                )
                new_batch_stats = mut["batch_stats"]

                logits_clean = state.apply_fn(
                    {"params": params, "batch_stats": new_batch_stats},
                    x, train=False
                )

                loss = optax.softmax_cross_entropy_with_integer_labels(logits_adv, y).mean()
                return loss, (new_batch_stats, logits_clean, logits_adv)

        elif defense == "trades":
            # Attack target from current model in eval mode, like official TRADES
            logits_nat_eval = state.apply_fn(frozen_vars, x, train=False)
            nat_probs_for_attack = jax.nn.softmax(logits_nat_eval, axis=-1)

            x_adv = trades_attack(
                apply_fn=state.apply_fn,
                params=state.params,
                batch_stats=state.batch_stats,
                x=x,
                nat_probs=nat_probs_for_attack,
                rng=attack_rng,
                dataset_name=dataset_name,
                steps=ATTACK_CFG["pgd_steps_train"],
            )

            def loss_fn(params):
                # First clean forward in train mode
                variables = {"params": params, "batch_stats": state.batch_stats}
                logits_clean, mut_clean = state.apply_fn(
                    variables, x, train=True, mutable=["batch_stats"]
                )
                batch_stats_after_clean = mut_clean["batch_stats"]

                # Then adversarial forward also in train mode
                logits_adv, mut_adv = state.apply_fn(
                    {"params": params, "batch_stats": batch_stats_after_clean},
                    x_adv,
                    train=True,
                    mutable=["batch_stats"],
                )
                new_batch_stats = mut_adv["batch_stats"]

                nat_loss = optax.softmax_cross_entropy_with_integer_labels(logits_clean, y).mean()
                robust_loss = kl_divergence_probs_logits(
                    jax.nn.softmax(logits_clean, axis=-1),
                    logits_adv,
                ).mean()

                loss = nat_loss + beta * robust_loss
                return loss, (new_batch_stats, logits_clean, logits_adv)

        elif defense == "mart":
            x_adv = pgd_ce_attack(
                apply_fn=state.apply_fn,
                params=state.params,
                batch_stats=state.batch_stats,
                x=x,
                y=y,
                rng=attack_rng,
                dataset_name=dataset_name,
                steps=ATTACK_CFG["pgd_steps_train"],
                alpha="train",
            )

            def loss_fn(params):
                variables = {"params": params, "batch_stats": state.batch_stats}
                logits_clean, mut = state.apply_fn(
                    variables, x, train=True, mutable=["batch_stats"]
                )
                batch_stats_after_clean = mut["batch_stats"]

                logits_adv = state.apply_fn(
                    {"params": params, "batch_stats": batch_stats_after_clean},
                    x_adv, train=False
                )

                loss = mart_loss(logits_clean, logits_adv, y, beta=beta)
                return loss, (batch_stats_after_clean, logits_clean, logits_adv)

        else:
            raise ValueError(defense)

        (loss, (new_batch_stats, logits_clean, logits_adv)), grads = jax.value_and_grad(
            loss_fn, has_aux=True
        )(state.params)

        grads = lax.pmean(grads, axis_name="batch")
        loss = lax.pmean(loss, axis_name="batch")
        clean_acc = lax.pmean(
            jnp.mean(jnp.argmax(logits_clean, axis=-1) == y), axis_name="batch"
        )
        adv_acc = lax.pmean(
            jnp.mean(jnp.argmax(logits_adv, axis=-1) == y), axis_name="batch"
        )
        new_batch_stats = lax.pmean(new_batch_stats, axis_name="batch")

        new_state = state.apply_gradients(grads=grads, batch_stats=new_batch_stats)
        metrics = {
            "loss": loss,
            "clean_acc": clean_acc,
            "adv_acc": adv_acc,
        }
        return new_state, metrics

    return train_step


def make_eval_step(dataset_name):
    @partial(jax.pmap, axis_name="batch")
    def eval_step(state, batch, rng):
        x = batch["images"]
        y = batch["labels"]

        x_adv = pgd_ce_attack(
            apply_fn=state.apply_fn,
            params=state.params,
            batch_stats=state.batch_stats,
            x=x,
            y=y,
            rng=rng,
            dataset_name=dataset_name,
            steps=ATTACK_CFG["pgd_steps_eval"],
            alpha="eval",
        )

        variables = {"params": state.params, "batch_stats": state.batch_stats}
        logits_clean = state.apply_fn(variables, x, train=False)
        logits_adv = state.apply_fn(variables, x_adv, train=False)

        clean_acc = lax.pmean(
            jnp.mean(jnp.argmax(logits_clean, axis=-1) == y), axis_name="batch"
        )
        robust_acc = lax.pmean(
            jnp.mean(jnp.argmax(logits_adv, axis=-1) == y), axis_name="batch"
        )

        return {
            "clean_acc": clean_acc,
            "robust_acc": robust_acc,
        }

    return eval_step

def scalarize_metrics(metrics_list):
    out = {}
    for key in metrics_list[0].keys():
        out[key] = float(np.mean([float(m[key]) for m in metrics_list]))
    return out

def unreplicate_state(pstate):
    return jax_utils.unreplicate(pstate)

def save_train_state(path: Path, state: TrainState):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "wb") as f:
        f.write(serialization.to_bytes(state))

def restore_train_state(path: Path, template_state: TrainState):
    with open(path, "rb") as f:
        return serialization.from_bytes(template_state, f.read())

def save_analysis_payload(path: Path, state: TrainState):
    payload = {
        "params": state.params,
        "batch_stats": state.batch_stats,
        "step": np.asarray(int(state.step), dtype=np.int32),
    }
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "wb") as f:
        f.write(serialization.to_bytes(payload))

print("Robust-training utilities ready.")

In [ ]:
# @title TRADES and MART loss definitions

# align MART beta with the usual default
TRAINING_CFG[("cifar10", "resnet18_mart")]["beta"] = 6.0
TRAINING_CFG[("cifar100", "resnet18_mart")]["beta"] = 6.0

def trades_attack(apply_fn, params, batch_stats, x, nat_probs, rng, dataset_name, steps):
    """
    TRADES-style inner attack:
    - tiny Gaussian init
    - maximize KL( p_nat || p_adv )
    """
    lower, upper, eps, alpha_train, _ = dataset_attack_tensors(dataset_name)

    std = jnp.array(DATASET_CFG[dataset_name]["std"], dtype=jnp.float32).reshape(1, 1, 1, 3)
    noise_scale = 0.001 / std
    noise = random.normal(rng, shape=x.shape) * noise_scale
    x_adv = jnp.clip(x + noise, lower, upper)

    variables = {"params": params, "batch_stats": batch_stats}
    nat_probs = lax.stop_gradient(nat_probs)

    def loss_on_x(x_in):
        logits_adv = apply_fn(variables, x_in, train=False)
        return kl_divergence_probs_logits(nat_probs, logits_adv).mean()

    def body_fn(i, x_adv_now):
        grad = jax.grad(loss_on_x)(x_adv_now)
        x_next = x_adv_now + alpha_train * jnp.sign(grad)
        x_next = clip_project(x_next, x, lower, upper, eps)
        return x_next

    return lax.fori_loop(0, steps, body_fn, x_adv)


def mart_attack(apply_fn, params, batch_stats, x, y, rng, dataset_name, steps):
    """
    MART-style inner attack:
    - tiny Gaussian init
    - maximize CE on adversarial example
    """
    lower, upper, eps, alpha_train, _ = dataset_attack_tensors(dataset_name)

    std = jnp.array(DATASET_CFG[dataset_name]["std"], dtype=jnp.float32).reshape(1, 1, 1, 3)
    noise_scale = 0.001 / std
    noise = random.normal(rng, shape=x.shape) * noise_scale
    x_adv = jnp.clip(x + noise, lower, upper)

    variables = {"params": params, "batch_stats": batch_stats}

    def loss_on_x(x_in):
        logits = apply_fn(variables, x_in, train=False)
        return optax.softmax_cross_entropy_with_integer_labels(logits, y).mean()

    def body_fn(i, x_adv_now):
        grad = jax.grad(loss_on_x)(x_adv_now)
        x_next = x_adv_now + alpha_train * jnp.sign(grad)
        x_next = clip_project(x_next, x, lower, upper, eps)
        return x_next

    return lax.fori_loop(0, steps, body_fn, x_adv)


def mart_loss(logits_clean, logits_adv, labels, beta):
    """
    MART loss close to the reference formulation.
    """
    batch_size = logits_clean.shape[0]

    adv_probs = jax.nn.softmax(logits_adv, axis=-1)

    top2 = jnp.argsort(adv_probs, axis=1)[:, -2:]
    new_y = jnp.where(top2[:, -1] == labels, top2[:, -2], top2[:, -1])

    loss_adv = (
        optax.softmax_cross_entropy_with_integer_labels(logits_adv, labels)
        - jnp.take_along_axis(
            jnp.log(jnp.clip(1.0001 - adv_probs, 1e-12, 1.0)),
            new_y[:, None],
            axis=1,
        ).squeeze(1)
    )

    nat_probs = jax.nn.softmax(logits_clean, axis=-1)
    true_probs = jnp.take_along_axis(nat_probs, labels[:, None], axis=1).squeeze(1)

    # KL(nat || adv), weighted by (1 - p_true_nat)
    kl_per_example = jnp.sum(
        nat_probs * (
            jnp.log(jnp.clip(nat_probs, 1e-12, 1.0))
            - jnp.log(jnp.clip(adv_probs, 1e-12, 1.0))
        ),
        axis=1,
    )

    loss_robust = jnp.sum(kl_per_example * (1.0000001 - true_probs)) / batch_size
    return jnp.mean(loss_adv) + beta * loss_robust


def make_train_step(dataset_name, model_name):
    _, defense = parse_model_name(model_name)
    beta = TRAINING_CFG[(dataset_name, model_name)]["beta"]

    @partial(jax.pmap, axis_name="batch", donate_argnums=(0,))
    def train_step(state, batch, rng):
        x = batch["images"]
        y = batch["labels"]

        attack_rng, _ = random.split(rng)
        frozen_vars = {"params": state.params, "batch_stats": state.batch_stats}

        if defense == "pgd":
            x_adv = pgd_ce_attack(
                apply_fn=state.apply_fn,
                params=state.params,
                batch_stats=state.batch_stats,
                x=x,
                y=y,
                rng=attack_rng,
                dataset_name=dataset_name,
                steps=ATTACK_CFG["pgd_steps_train"],
                alpha="train",
            )

            def loss_fn(params):
                variables = {"params": params, "batch_stats": state.batch_stats}
                logits_adv, mut = state.apply_fn(
                    variables, x_adv, train=True, mutable=["batch_stats"]
                )
                new_batch_stats = mut["batch_stats"]

                logits_clean = state.apply_fn(
                    {"params": params, "batch_stats": new_batch_stats},
                    x, train=False
                )

                loss = optax.softmax_cross_entropy_with_integer_labels(logits_adv, y).mean()
                return loss, (new_batch_stats, logits_clean, logits_adv)

        elif defense == "trades":
            # Natural prediction for the inner KL attack in eval mode
            logits_nat_eval = state.apply_fn(frozen_vars, x, train=False)
            nat_probs_for_attack = jax.nn.softmax(logits_nat_eval, axis=-1)

            x_adv = trades_attack(
                apply_fn=state.apply_fn,
                params=state.params,
                batch_stats=state.batch_stats,
                x=x,
                nat_probs=nat_probs_for_attack,
                rng=attack_rng,
                dataset_name=dataset_name,
                steps=ATTACK_CFG["pgd_steps_train"],
            )

            def loss_fn(params):
                # Clean forward in train mode
                variables = {"params": params, "batch_stats": state.batch_stats}
                logits_clean, mut_clean = state.apply_fn(
                    variables, x, train=True, mutable=["batch_stats"]
                )
                batch_stats_after_clean = mut_clean["batch_stats"]

                # Adversarial forward also in train mode
                logits_adv, mut_adv = state.apply_fn(
                    {"params": params, "batch_stats": batch_stats_after_clean},
                    x_adv,
                    train=True,
                    mutable=["batch_stats"],
                )
                new_batch_stats = mut_adv["batch_stats"]

                nat_loss = optax.softmax_cross_entropy_with_integer_labels(logits_clean, y).mean()
                robust_loss = kl_divergence_probs_logits(
                    jax.nn.softmax(logits_clean, axis=-1),
                    logits_adv,
                ).mean()

                loss = nat_loss + beta * robust_loss
                return loss, (new_batch_stats, logits_clean, logits_adv)

        elif defense == "mart":
            x_adv = mart_attack(
                apply_fn=state.apply_fn,
                params=state.params,
                batch_stats=state.batch_stats,
                x=x,
                y=y,
                rng=attack_rng,
                dataset_name=dataset_name,
                steps=ATTACK_CFG["pgd_steps_train"],
            )

            def loss_fn(params):
                # Clean forward in train mode
                variables = {"params": params, "batch_stats": state.batch_stats}
                logits_clean, mut_clean = state.apply_fn(
                    variables, x, train=True, mutable=["batch_stats"]
                )
                batch_stats_after_clean = mut_clean["batch_stats"]

                # Adversarial forward ALSO in train mode
                logits_adv, mut_adv = state.apply_fn(
                    {"params": params, "batch_stats": batch_stats_after_clean},
                    x_adv,
                    train=True,
                    mutable=["batch_stats"],
                )
                new_batch_stats = mut_adv["batch_stats"]

                loss = mart_loss(logits_clean, logits_adv, y, beta=beta)
                return loss, (new_batch_stats, logits_clean, logits_adv)

        else:
            raise ValueError(defense)

        (loss, (new_batch_stats, logits_clean, logits_adv)), grads = jax.value_and_grad(
            loss_fn, has_aux=True
        )(state.params)

        grads = lax.pmean(grads, axis_name="batch")
        loss = lax.pmean(loss, axis_name="batch")
        clean_acc = lax.pmean(
            jnp.mean(jnp.argmax(logits_clean, axis=-1) == y), axis_name="batch"
        )
        adv_acc = lax.pmean(
            jnp.mean(jnp.argmax(logits_adv, axis=-1) == y), axis_name="batch"
        )
        new_batch_stats = lax.pmean(new_batch_stats, axis_name="batch")

        new_state = state.apply_gradients(grads=grads, batch_stats=new_batch_stats)
        metrics = {
            "loss": loss,
            "clean_acc": clean_acc,
            "adv_acc": adv_acc,
        }
        return new_state, metrics

    return train_step

print("Patched TRADES + MART.")

In [ ]:
# @title 6. Single-run trainer with robust-best checkpointing

def train_one_run(dataset_name: str, model_name: str, seed: int,
                  save_root=None, run_group=None, verbose=True):
    if save_root is None:
        save_root = RUN_ROOT
    if run_group is None:
        run_group = RUN_GROUP

    set_all_seeds(seed)

    sched = TRAINING_CFG[(dataset_name, model_name)]
    rdir = run_dir(save_root, run_group, dataset_name, model_name, seed)
    rdir.mkdir(parents=True, exist_ok=True)

    done_path = rdir / "done.json"
    if done_path.exists():
        if verbose:
            print("Already finished:", rdir)
        return load_json(done_path)

    if verbose:
        print(f"\n=== {dataset_name} | {model_name} | seed={seed} ===")
        print("Run dir:", rdir)

    train_iter, eval_ds, steps_per_epoch, eval_steps, class_names = make_dataloaders(
        dataset_name, batch_size=sched["batch_size"], seed=seed
    )

    model, model_cfg = build_model(model_name, dataset_name)
    layer_names = get_layer_names(model, dataset_name)

    state, lr_fn = init_state_for_run(
        model, dataset_name, model_name, seed, steps_per_epoch
    )

    meta = {
        "dataset_name": dataset_name,
        "model_name": model_name,
        "seed": seed,
        "dataset_cfg": DATASET_CFG[dataset_name],
        "model_cfg": model_cfg,
        "attack_cfg": ATTACK_CFG,
        "schedule": sched,
        "layer_names": layer_names,
        "class_names": class_names,
        "created_at_unix": time.time(),
    }
    meta_path = rdir / "meta.json"
    if not meta_path.exists():
        save_json(meta_path, meta)

    full_state_path = rdir / "last_train_state.msgpack"
    run_state_path = rdir / "run_state.json"
    history_path = rdir / "history.csv"

    start_epoch = 1
    best_clean_acc = -1.0
    best_robust_acc = -1.0
    best_epoch = -1
    history = []

    if full_state_path.exists() and run_state_path.exists():
        if verbose:
            print("Resuming from:", full_state_path)
        state = restore_train_state(full_state_path, state)
        rs = load_json(run_state_path)
        start_epoch = int(rs["epoch_completed"]) + 1
        best_clean_acc = float(rs["best_clean_acc"])
        best_robust_acc = float(rs["best_robust_acc"])
        best_epoch = int(rs["best_epoch"])
        if history_path.exists():
            history = pd.read_csv(history_path).to_dict("records")

    pstate = jax_utils.replicate(state)
    p_train_step = make_train_step(dataset_name=dataset_name, model_name=model_name)
    p_eval_step = make_eval_step(dataset_name=dataset_name)

    master_rng = random.PRNGKey(seed + 12345)

    for epoch in range(start_epoch, sched["epochs"] + 1):
        t0 = time.time()

        # train
        train_metrics = []
        for _ in tqdm(range(steps_per_epoch), disable=not verbose, leave=False, desc=f"train e{epoch:03d}"):
            batch = next(train_iter)
            master_rng, step_rng = random.split(master_rng)
            step_rngs = random.split(step_rng, NUM_DEVICES)
            pstate, metrics = p_train_step(pstate, batch, step_rngs)
            metrics = jax.device_get(jax_utils.unreplicate(metrics))
            train_metrics.append(metrics)

        train_log = scalarize_metrics(train_metrics)

        # eval
        eval_metrics = []
        eval_iter = numpy_prefetch_iter(eval_ds, prefetch_size=2)
        for _ in tqdm(range(eval_steps), disable=not verbose, leave=False, desc=f"eval  e{epoch:03d}"):
            batch = next(eval_iter)
            master_rng, eval_rng = random.split(master_rng)
            eval_rngs = random.split(eval_rng, NUM_DEVICES)
            metrics = p_eval_step(pstate, batch, eval_rngs)
            metrics = jax.device_get(jax_utils.unreplicate(metrics))
            eval_metrics.append(metrics)

        eval_log = scalarize_metrics(eval_metrics)
        host_state = unreplicate_state(pstate)

        save_train_state(rdir / "last_train_state.msgpack", host_state)
        save_analysis_payload(rdir / "last_analysis.msgpack", host_state)

        improved = eval_log["robust_acc"] > best_robust_acc
        if improved:
            best_clean_acc = eval_log["clean_acc"]
            best_robust_acc = eval_log["robust_acc"]
            best_epoch = epoch
            save_analysis_payload(rdir / "best_analysis.msgpack", host_state)
            save_json(
                rdir / "best_metrics.json",
                {
                    "best_clean_acc": best_clean_acc,
                    "best_robust_acc": best_robust_acc,
                    "best_epoch": best_epoch,
                    "step": int(host_state.step),
                },
            )

        row = {
            "epoch": epoch,
            "step": int(host_state.step),
            "lr": float(lr_fn(int(host_state.step))),
            "train_loss": train_log["loss"],
            "train_clean_acc": train_log["clean_acc"],
            "train_adv_acc": train_log["adv_acc"],
            "eval_clean_acc": eval_log["clean_acc"],
            "eval_robust_acc": eval_log["robust_acc"],
            "best_clean_acc_so_far": best_clean_acc,
            "best_robust_acc_so_far": best_robust_acc,
            "epoch_seconds": time.time() - t0,
        }
        history.append(row)
        pd.DataFrame(history).to_csv(history_path, index=False)

        save_json(
            run_state_path,
            {
                "epoch_completed": epoch,
                "best_clean_acc": best_clean_acc,
                "best_robust_acc": best_robust_acc,
                "best_epoch": best_epoch,
                "last_step": int(host_state.step),
            },
        )

        if verbose:
            print(
                f"[{dataset_name} | {model_name} | seed={seed:02d}] "
                f"epoch {epoch:03d}/{sched['epochs']}  "
                f"train_clean={row['train_clean_acc']:.4f}  "
                f"train_adv={row['train_adv_acc']:.4f}  "
                f"eval_clean={row['eval_clean_acc']:.4f}  "
                f"eval_robust={row['eval_robust_acc']:.4f}  "
                f"best_robust={best_robust_acc:.4f}"
            )

        gc.collect()

    result = {
        "run_dir": str(rdir),
        "dataset_name": dataset_name,
        "model_name": model_name,
        "seed": seed,
        "best_clean_acc": best_clean_acc,
        "best_robust_acc": best_robust_acc,
        "best_epoch": best_epoch,
        "history_path": str(history_path),
    }
    save_json(done_path, result)

    if verbose:
        print("Finished:", rdir)
        print("Best robust acc:", best_robust_acc, "at epoch", best_epoch)

    return result

print("Single-run trainer ready.")

In [ ]:
# @title 7. Launcher for the 30-model adversarial bank

ONLY_DATASETS = None          # e.g. ["cifar10"]
ONLY_MODELS = None            # e.g. ["resnet18_pgd"]
ONLY_SEEDS = None             # e.g. [0, 1]
MAX_RUNS_THIS_SESSION = None  # e.g. 2

def run_is_finished(dataset_name, model_name, seed, save_root=None, run_group=None):
    if save_root is None:
        save_root = RUN_ROOT
    if run_group is None:
        run_group = RUN_GROUP
    rdir = run_dir(save_root, run_group, dataset_name, model_name, seed)
    return (rdir / "done.json").exists()

grid = []
for dataset_name in DATASET_NAMES:
    if ONLY_DATASETS is not None and dataset_name not in ONLY_DATASETS:
        continue
    for model_name in MODEL_NAMES:
        if ONLY_MODELS is not None and model_name not in ONLY_MODELS:
            continue
        for seed in SEEDS:
            if ONLY_SEEDS is not None and seed not in ONLY_SEEDS:
                continue
            grid.append((dataset_name, model_name, seed))

print("Planned runs:", len(grid))
for item in grid[:10]:
    print(" ", item)
if len(grid) > 10:
    print(" ...")

completed = 0
launched = 0

for dataset_name, model_name, seed in grid:
    if run_is_finished(dataset_name, model_name, seed):
        print(f"SKIP done: {dataset_name} | {model_name} | seed={seed}")
        completed += 1
        continue

    train_one_run(
        dataset_name=dataset_name,
        model_name=model_name,
        seed=seed,
        save_root=RUN_ROOT,
        run_group=RUN_GROUP,
        verbose=True,
    )
    launched += 1

    if MAX_RUNS_THIS_SESSION is not None and launched >= MAX_RUNS_THIS_SESSION:
        print(f"Stopping because MAX_RUNS_THIS_SESSION={MAX_RUNS_THIS_SESSION}")
        break

print("Completed already:", completed)
print("Launched now:", launched)